# Automotive RAG Question Answering System - Master Notebook

Welcome! This unified notebook executes the entire Automotive RAG System from end to end. It is designed to be run directly inside **Google Colab (T4 GPU)**.

### Pipeline Overview:
* **Phase 1:** Environment Setup, Document Upload, Extraction, Chunking, and FAISS Vector Indexing.
* **Phase 2:** RAG Execution (Retrieval, Prompting, and LLM Inference using 4-bit Quantization).
* **Phase 3:** Automated Benchmarking & Context Window Evaluation Study.

## Step 1: Environment Setup
First, we clone the backend repository, install the dependencies, and configure the Python path so our modules load correctly.

In [ ]:
import os
import sys

# Clone the repository if not already present
if not os.path.exists('Automotive-RAG-QA-System'):
    !git clone https://github.com/Adityakumar001-usn/Automotive-RAG-QA-System.git

os.chdir('Automotive-RAG-QA-System')
sys.path.append(os.getcwd())

# Install core requirements and system OCR dependencies
!apt-get install -y tesseract-ocr > /dev/null
!pip install -r requirements.txt > /dev/null

print("✅ Environment Setup Complete!")

## Step 2: Phase 1 - Upload Documents & Build Index
Upload your raw Automotive PDFs, CSVs, or TXT files. The system will extract the text, clean it, chunk it, and save the embeddings offline to a CPU FAISS index.

In [ ]:
from src.processing import process_document
from src.chunking import recursive_chunking
from src.embeddings import generate_embeddings
from src.vector_store import VectorStore
import shutil

try:
    from google.colab import files
    print("Please upload your automotive documents:")
    uploaded = files.upload()
    os.makedirs('uploaded_docs', exist_ok=True)
    
    file_paths = []
    for filename, data in uploaded.items():
        path = os.path.join('uploaded_docs', filename)
        with open(path, 'wb') as f:
            f.write(data)
        file_paths.append(path)
except ImportError:
    print("Not in Colab. Using test assets.")
    file_paths = ['test_assets/sample.pdf', 'test_assets/sample.txt']

all_chunks, all_metadatas = [], []
for path in file_paths:
    text, metadata = process_document(path)
    chunks = recursive_chunking(text)
    all_chunks.extend(chunks)
    all_metadatas.extend([metadata.copy() for _ in chunks])

if all_chunks:
    embeddings = generate_embeddings(all_chunks)
    vs = VectorStore()
    vs.build_index(embeddings, all_chunks, all_metadatas)
    vs.save_index('faiss_index')
    print("\n✅ Phase 1 Successful! Database built and saved to disk.")
else:
    print("No chunks generated.")

## Step 3: Phase 2 - RAG QA System Execution
With the database built, we initialize the `AutomotiveRAG` engine. The LLM (Phi-3) will load natively into your Colab T4 GPU using 4-bit quantization to prevent OOM errors. Ask a question to see the glass-box traceability.

In [ ]:
from src.retriever import Retriever
from src.rag_engine import AutomotiveRAG
from src.rag_evaluator import RAGEvaluator

rag = AutomotiveRAG(Retriever(vs))
evaluator = RAGEvaluator()

question = "What is the content of the PDF?"
result = rag.ask(question)

print("\n===========================")
print("🤖 AI ANSWER:")
print(result['answer'])
print("===========================")
print("\n📚 SOURCES USED:")
for s in result['sources']:
    print(f"- Document: {s['document_name']} (Chunk {s['chunk_id']}) | Distance: {s['distance']:.2f}")

## Step 4: Phase 3 - Context Window Benchmarking
Finally, we execute the automated evaluation study spanning dozens of questions across `512, 1024, 2048, 4096` token windows natively testing the limits of the hardware and RAG logic.

In [ ]:
from src.benchmark_runner import BenchmarkRunner
from IPython.display import display, Markdown, Image

runner = BenchmarkRunner(rag, questions_path='data/evaluation_questions.json')
runner.run()
runner.generate_outputs()
print("\n✅ Phase 3 Complete! Generating Analytics Dashboard...")

In [ ]:
display(Image(filename="results/comparison_dashboard.png"))
display(Markdown(open("results/phase3_analysis.md").read()))